# 20 — A Threshold With a Guarantee That Survives Selection

Sections 8.2 and 8.3 choose an operating point by targeting an empirical false-positive rate on
held-out buffer labels, then report a binomial bound from those same labels. That bound is not
valid: the threshold was selected using the observations the bound is computed from, so the
selection is unaccounted for. Targeting an empirical rate and controlling the population rate
with high probability are different claims.

This notebook replaces the construction with one whose guarantee holds despite the selection.
An order-statistic (conformal) threshold would fix the problem for continuous scores, but not
here: tree ensembles tie heavily, and simulation shows that construction failing in 8 to 15% of
draws once scores take few distinct values, against a 5% target. We therefore split the
calibration labels. The threshold is chosen on one half and certified on the other, so on the
certification half the false-positive count is Binomial with the population rate at a threshold
that was fixed before those labels were seen, and a Clopper-Pearson bound applies exactly. The
cost is that only half the benign labels certify, so the bound is wider. Simulation puts the
violation rate below 1% for every score granularity tested.

Three quantities are recorded per cell: the false positives observed on the certification half,
the benign count there, and the false-positive rate the threshold realises on evaluation data.
The guarantee is then checked rather than asserted, by measuring how often the realised rate
exceeds its own bound.

For comparison each cell also reports the previous empirical-target threshold and its
Clopper-Pearson bound, so the difference between the two constructions is visible.

Results append to fc_conformal_threshold.csv.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'

CFG = dict(corpora=['nf2018v2','nfunswv2','nftonv2','nfbotv2'], seed=42, test_size=0.30,
           eval_cap=200_000, pool_cap=200_000, budgets=[0.0001, 0.001, 0.01],
           div_budgets=[0.0001, 0.001], thr_frac=0.30, alpha=0.01, conf=0.95,
           rf_estimators=300, mlp_hidden=(128,64), mlp_max_iter=100, ece_bins=15)
MODELS = ['rf', 'lgbm', 'mlp']
CSV = f'{RESULT}/fc_conformal_threshold.csv'
COLS = ['seed','target','model','budget','rule','n_labels','n_fit','n_cal','n_benign_cal',
        'n_benign_select','n_benign_certify','fp_certify','thr_split','bound_split',
        'fpr_split','tpr_split','mcc_split',
        'thr_empirical','fp_obs_empirical','bound_naive','fpr_empirical','tpr_empirical','fit_s']
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label','Attack')]
print('features:', len(FEATURES))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(den > 0, num / den, 0.0)
    return out

def best_threshold_mcc(y_true, p_pos, n_grid=199):
    """Max MCC over a quantile grid of thresholds; robust to degenerate score distributions."""
    y = np.asarray(y_true).astype(int); p = np.asarray(p_pos)
    thr = np.unique(np.quantile(p, np.linspace(0.001, 0.999, n_grid)))
    if len(thr) < 2:
        thr = np.array([thr[0]]) if len(thr) else np.array([0.5])
    P = y.sum(); N = len(y) - P
    tp = np.array([(y[p >= t]).sum() for t in thr], dtype=float)
    fp = np.array([(p >= t).sum() for t in thr], dtype=float) - tp
    fn = P - tp; tn = N - fp
    mccs = mcc_from_counts(tp, fp, fn, tn)
    i = int(np.argmax(mccs))
    return float(mccs[i]), float(thr[i])

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, 0.0)

def best_threshold_mcc(y_true, p_pos):
    y = np.asarray(y_true).astype(np.int64); p = np.asarray(p_pos, dtype=np.float64)
    order = np.argsort(-p, kind='mergesort'); ps, ys = p[order], y[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last].astype(float), fp[last].astype(float), ps[last]
    mccs = mcc_from_counts(tp, fp, P - tp, N - fp)
    i = int(np.argmax(mccs))
    return (0.0, float('inf')) if mccs[i] <= 0 else (float(mccs[i]), float(cuts[i]))

def best_threshold_two_sided(y_true, p_pos):
    """Max MCC over both orientations. Returns (mcc, thr, orient) with orient +1 (p>=thr) or -1 ((1-p)>=thr)."""
    m_pos, t_pos = best_threshold_mcc(y_true, p_pos)
    m_neg, t_neg = best_threshold_mcc(y_true, 1.0 - np.asarray(p_pos))
    return (m_pos, t_pos, 1) if m_pos >= m_neg else (m_neg, t_neg, -1)

def apply_oriented(p_pos, thr, orient):
    p = np.asarray(p_pos)
    return ((p if orient == 1 else 1.0 - p) >= thr).astype(int)

def mcc_of_pred(y_true, pred):
    return matthews_corrcoef(np.asarray(y_true), np.asarray(pred))

# ---------- acquisition rules: return integer positions into the pool ----------
def acquire_uniform(n_pool, k, seed):
    return np.random.default_rng(seed).choice(n_pool, size=min(k, n_pool), replace=False)

def acquire_uncertainty(p_pool, k):
    p = np.clip(np.asarray(p_pool), 1e-9, 1 - 1e-9)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    return np.argsort(-ent, kind='mergesort')[:k]

def acquire_diversity(X_pool, k, seed):
    """k-means with k clusters on standardised features; per cluster, the member nearest its centroid.
    Distances are computed to each row's own centroid only (O(n*features)), never as an n x k matrix."""
    Xs = StandardScaler().fit_transform(np.asarray(X_pool, dtype=np.float64))
    k = min(k, len(Xs))
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=4096, n_init=1, max_iter=50).fit(Xs)
    labels = km.labels_
    own = np.einsum('ij,ij->i', Xs - km.cluster_centers_[labels], Xs - km.cluster_centers_[labels])
    df = pd.DataFrame({'lab': labels, 'd': own})
    chosen = df.groupby('lab').d.idxmin().values.astype(int)
    if len(chosen) < k:
        rest = np.setdiff1d(np.arange(len(Xs)), chosen)
        chosen = np.concatenate([chosen, rest[np.argsort(own[rest])[:k - len(chosen)]]])
    return chosen

def acquire_hybrid(X_pool, p_pool, k, seed, factor=5):
    cand = acquire_uncertainty(p_pool, min(len(p_pool), factor * k))
    sub = acquire_diversity(np.asarray(X_pool)[cand], k, seed)
    return cand[sub]

# ---------- cross-validated buffer-only estimate on the buffer itself ----------
def cv_estimate(make_model_fn, Xb, yb, seed, n_splits=3):
    """Mean MCC over stratified folds; returns 0.0 when a class has fewer than n_splits rows."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2 or np.bincount(yb).min() < n_splits:
        return 0.0
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for tr, te in skf.split(Xb, yb):
        if len(np.unique(yb[tr])) < 2:
            out.append(0.0); continue
        mdl = make_model_fn(len(tr)); mdl.fit(Xb.iloc[tr], yb[tr])
        out.append(mcc_of_pred(yb[te], (mdl.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
    return float(np.mean(out))

In [ ]:
import numpy as np, pandas as pd

def tpr_at_fpr(y, p, target_fpr):
    """Highest TPR attainable at or below target_fpr, with the threshold that attains it.
    Thresholds are evaluated at every distinct score, so the result is exact."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    order = np.argsort(-p, kind='mergesort'); ys, ps = y[order], p[order]
    P = int(ys.sum()); N = len(ys) - P
    if P == 0 or N == 0:
        return np.nan, np.nan
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last], fp[last], ps[last]
    ok = (fp / N) <= target_fpr
    if not ok.any():
        return 0.0, float(cuts[0]) + 1e-12
    i = int(np.argmax(np.where(ok, tp, -1)))
    return float(tp[i] / P), float(cuts[i])

def threshold_for_fpr_on_buffer(y_buf, p_buf, target_fpr):
    """Operating point an analyst could actually set: chosen on the labelled buffer only."""
    _, thr = tpr_at_fpr(y_buf, p_buf, target_fpr)
    return thr

def realised_at_threshold(y, p, thr):
    """TPR and FPR on the evaluation set at an externally chosen threshold."""
    y = np.asarray(y).astype(int); pred = (np.asarray(p) >= thr).astype(int)
    P = int(y.sum()); N = len(y) - P
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    return (tp / P if P else np.nan), (fp / N if N else np.nan)

def family_holdout_buffer(train_full, held_family, frac, seed, min_per_group=1):
    """Stratified buffer drawn only from families other than held_family, so the retrained
    model has never seen that attack type. Size matches the ordinary buffer at this budget."""
    pool = train_full[train_full['Attack'] != held_family]
    k = max(1, int(round(len(train_full) * frac)))
    parts = []
    for fam, g in pool.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * k / len(pool)))))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def eligible_families(df, min_rows=2000, max_share=0.60):
    """Attack families large enough to matter but not so dominant that holding one out
    leaves nothing to train on."""
    v = df.loc[df['Attack'] != 'Benign', 'Attack'].value_counts()
    n_att = int((df['Attack'] != 'Benign').sum())
    return [f for f, c in v.items() if c >= min_rows and c / n_att <= max_share]

def metrics_at_threshold(y_true, p_pos, thr):
    """Threshold-sensitive metrics taken at an externally chosen cut rather than at 0.5.
    Needed because the rethreshold strategy does not operate at 0.5, so reporting its
    macro-F1 or false-positive rate from the default cut would describe a different detector."""
    from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
    y = np.asarray(y_true).astype(int)
    pred = (np.asarray(p_pos) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(mcc=matthews_corrcoef(y, pred),
                macro_f1=f1_score(y, pred, average='macro'),
                fp_rate=fp / (fp + tn) if (fp + tn) else np.nan)

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, matthews_corrcoef

def split_buffer(buf, thr_frac, seed, label_col='Label'):
    """Split a labelled buffer into a part used to fit the model and a part reserved for
    choosing the operating threshold. Both parts are drawn from the same budget, so the
    total label count is unchanged: thresholds are no longer selected on training rows."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(buf))
    thr_idx = []
    for _, g in buf.groupby(label_col, sort=True):
        pos = np.where(buf[label_col].values == g[label_col].iloc[0])[0]
        k = max(1, int(round(len(pos) * thr_frac))) if len(pos) > 1 else 0
        if k:
            thr_idx.extend(rng.choice(pos, size=min(k, len(pos) - 1), replace=False))
    thr_idx = np.array(sorted(set(thr_idx)), dtype=int)
    fit_idx = np.setdiff1d(idx, thr_idx)
    return buf.iloc[fit_idx], buf.iloc[thr_idx]

def orientation_signs(y, p):
    """The three quantities that must be distinguished: the sign of the score-label
    covariance, the sign of AUROC - 0.5, and which threshold orientation attains the higher
    MCC. They are not equivalent, so each is measured rather than inferred from another."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    # named cov_sy rather than cov: a column called 'cov' on a DataFrame is shadowed by
    # the DataFrame.cov method under attribute access, which silently yields the method
    out = dict(cov_sy=np.nan, auroc=np.nan, mcc_inc=np.nan, mcc_dec=np.nan)
    if len(np.unique(y)) < 2 or p.std() < 1e-12:
        return out
    out['cov_sy'] = float(np.cov(p, y, bias=True)[0, 1])
    out['auroc'] = float(roc_auc_score(y, p))
    order = np.argsort(-p, kind='mergesort'); ys = y[order]; ps = p[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp = tp[last], fp[last]
    def mcc(tp, fp, fn, tn):
        d = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        return np.where(d > 0, (tp * tn - fp * fn) / np.maximum(d, 1e-300), 0.0)
    out['mcc_inc'] = float(np.max(mcc(tp, fp, P - tp, N - fp)))
    tp2, fp2 = P - tp, N - fp
    out['mcc_dec'] = float(np.max(mcc(tp2, fp2, tp, fp)))
    return out

def tpr_at_common_fpr(y, p, cap):
    """Highest TPR attainable at or below a false-positive rate measured on the evaluation
    set itself. This is an oracle diagnostic, not a deployable threshold rule."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    P, N = int(y.sum()), int((1 - y).sum())
    if P == 0 or N == 0:
        return np.nan
    order = np.argsort(-p, kind='mergesort'); ys = y[order]; ps = p[order]
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp = tp[last], fp[last]
    ok = (fp / N) <= cap
    return float(tp[ok].max() / P) if ok.any() else 0.0

def uniform_buffer(train_full, frac, seed):
    """A label-blind sample: rows are drawn at random from the unlabelled pool with no use
    of family or class information, which is what an operator can actually do before paying
    for labels. The family-stratified buffer used elsewhere requires knowing attack families
    in advance and is therefore an oracle reference."""
    k = max(1, int(round(len(train_full) * frac)))
    return train_full.sample(n=min(k, len(train_full)), random_state=seed)

In [ ]:
import numpy as np

def label_counts(df, label_col='Label'):
    """Benign and attack counts, reported per split because the benign count in the
    threshold portion, not the buffer size, is what limits operating-point estimation."""
    y = df[label_col].values
    return int((y == 0).sum()), int((y == 1).sum())

def zero_event_upper(n, conf=0.95):
    """One-sided upper bound on a rate after observing no events in n trials. With two
    benign examples and no false positives the true rate is bounded only below 78%."""
    return float(1 - (1 - conf) ** (1.0 / n)) if n > 0 else float('nan')

In [ ]:
import numpy as np, pandas as pd

def reserve_then_acquire(pool, k_total, thr_frac, seed, acquire_fn=None, features=None, clean_fn=None):
    """Allocate the sampling roles before acquisition rather than after.

    A fraction thr_frac of the budget is drawn uniformly at random from the pool and reserved
    for choosing the operating threshold, so that sample is representative of target traffic.
    The remainder is acquired from the pool with those rows removed, by whatever rule is passed.
    Total purchased labels equal k_total either way, so the budget is unchanged; what changes is
    that the threshold sample is no longer drawn from the same selection as the training sample.

    Returns (fit_rows, threshold_rows)."""
    k_thr = max(1, int(round(k_total * thr_frac)))
    k_fit = max(1, k_total - k_thr)
    reserved = pool.sample(n=min(k_thr, len(pool)), random_state=seed)
    remaining = pool.drop(index=reserved.index)
    if acquire_fn is None:
        fit = remaining.sample(n=min(k_fit, len(remaining)), random_state=seed)
    else:
        X, _ = clean_fn(remaining, features)
        idx = acquire_fn(X.values, min(k_fit, len(remaining)), seed)
        fit = remaining.iloc[idx]
    return fit, reserved

def observed_fp(y_thr, p_thr, thr):
    """False positives actually observed on the threshold sample at the chosen cut, with the
    benign count, so a bound can be computed per procedure rather than from a mean sample size."""
    y = np.asarray(y_thr).astype(int); p = np.asarray(p_thr)
    benign = (y == 0)
    n = int(benign.sum())
    if n == 0 or np.isnan(thr):
        return 0, 0
    return int((p[benign] >= thr).sum()), n

def clopper_upper(k, n, conf=0.95):
    """One-sided upper confidence bound on a rate after observing k events in n trials.
    Reduces to 1 - (1-conf)^(1/n) when k = 0, and is reported to three decimals rather than
    rounded to whole percent, since a bound of 0.054% is not the same claim as 0%."""
    if n == 0:
        return float('nan')
    from scipy.stats import beta
    if k >= n:
        return 1.0
    return float(beta.ppf(conf, k + 1, n - k))

In [ ]:
import numpy as np
from scipy.stats import beta

def split_certify(y_cal, p_cal, alpha, seed, select_frac=0.5):
    """Choose the threshold on one half of the calibration labels and certify it on the other.

    Targeting an empirical error rate and then bounding it on the same labels is invalid: the
    threshold was chosen using the observations the bound is built from. An order-statistic
    (conformal) construction fixes that for continuous scores, but not here: tree ensembles tie
    heavily, and simulation shows the order-statistic bound failing in 8 to 15% of draws once
    scores take few distinct values, against a 5% target.

    Sample splitting is valid whatever the score distribution. The threshold is a function of the
    selection half alone, so on the certification half the false-positive count is Binomial(n, p)
    with p the population rate at that fixed threshold, and a Clopper-Pearson upper bound applies
    exactly. The cost is that only half the benign labels certify, so the bound is wider.

    Returns (threshold, fp_certify, n_benign_certify, n_benign_select)."""
    y = np.asarray(y_cal).astype(int); p = np.asarray(p_cal, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    cut = int(round(len(y) * select_frac))
    sel, cert = idx[:cut], idx[cut:]
    ys, ps = y[sel], p[sel]
    yc, pc = y[cert], p[cert]
    nb_sel = int((ys == 0).sum()); nb_cert = int((yc == 0).sum())
    if nb_sel == 0:
        return np.nan, 0, nb_cert, nb_sel
    ben = np.sort(ps[ys == 0])[::-1]
    budget = int(np.floor(alpha * (nb_sel + 1)))
    vals = np.unique(ben)[::-1]
    thr = None
    for v in vals:
        if int((ben >= v).sum()) <= budget:
            thr = float(v); break
    if thr is None:
        thr = float(vals[0]) + max(np.spacing(float(vals[0])), 1e-12)
    fp = int((pc[yc == 0] >= thr).sum()) if nb_cert else 0
    return thr, fp, nb_cert, nb_sel

def clopper_pearson_upper(k, n, conf=0.95):
    """One-sided upper bound on a rate after k events in n trials, valid when the threshold was
    fixed independently of these n observations. k = 0 reduces to 1 - (1-conf)^(1/n)."""
    if n == 0:
        return float('nan')
    if k >= n:
        return 1.0
    return float(beta.ppf(conf, k + 1, n - k))

In [ ]:
from sklearn.model_selection import train_test_split

def record(row):
    pd.DataFrame([{c: row.get(c, np.nan) for c in COLS}], columns=COLS).to_csv(
        CSV, mode='a', index=False, header=not os.path.exists(CSV))

def key(tgt, m, b, rule): return (tgt, m, f'{float(b):.6g}', rule)

done = set()
if os.path.exists(CSV):
    prev = pd.read_csv(CSV)
    done = set(key(r.target, r.model, r.budget, r.rule) for r in prev.itertuples())
    print(f'resume: {len(done)} rows recorded')

def is_done(*k): return key(*k) in done

def mark(tgt, m, b, rule, **kw):
    record(dict(seed=CFG['seed'], target=tgt, model=m, budget=b, rule=rule, **kw))
    done.add(key(tgt, m, b, rule))
    print(f"  {tgt} {m} b={b} {rule}: fp={kw.get('fp_certify',-1)}/{kw.get('n_benign_certify',-1)} "
          f"bound={kw.get('bound_split', float('nan')):.4f} "
          f"realised={kw.get('fpr_split', float('nan')):.4f} "
          f"{'OK' if kw.get('fpr_split', 1) <= kw.get('bound_split', 0) else 'EXCEEDS'}")

T0 = time.time()
def el(): return f'[{(time.time()-T0)/60:5.1f}m]'

seed = CFG['seed']
parts = {}
for tag, d in DATASETS.items():
    tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
    tr = tr.reset_index(drop=True)
    parts[tag] = dict(train_full=tr,
                      eval=stratified_cap(te, CFG['eval_cap'], seed).reset_index(drop=True),
                      pool=stratified_cap(tr, CFG['pool_cap'], seed).reset_index(drop=True))

# diversity is not run at every budget, so the check must only count combinations that exist
def _planned():
    for t_ in CFG['corpora']:
        for b_ in CFG['budgets']:
            for r_ in ('diversity','uniform','stratified'):
                if r_ == 'diversity' and b_ not in CFG['div_budgets']:
                    continue
                for m_ in MODELS:
                    yield t_, m_, b_, r_
need = any(not is_done(*k) for k in _planned())
BUF = {}
for tgt in (CFG['corpora'] if need else []):
    pool = parts[tgt]['pool']
    Xpool, _ = clean_X(pool, FEATURES)
    for b in CFG['budgets']:
        k = max(1, int(round(len(parts[tgt]['train_full']) * b)))
        BUF[(tgt, b, 'stratified')] = stratified_frac(parts[tgt]['train_full'], b, seed)
        BUF[(tgt, b, 'uniform')]    = uniform_buffer(parts[tgt]['train_full'], b, seed)
        if b in CFG['div_budgets']:
            BUF[(tgt, b, 'diversity')] = pool.iloc[acquire_diversity(Xpool.values, min(k, len(pool)), seed)]
    print(f'{el()} {tgt}: buffers built')
    del Xpool; gc.collect()

for tgt in CFG['corpora']:
    ev = parts[tgt]['eval']; yev = ev['Label'].values
    for mname in MODELS:
        for b in CFG['budgets']:
            for rule in ['diversity', 'uniform', 'stratified']:
                if (tgt, b, rule) not in BUF or is_done(tgt, mname, b, rule):
                    continue
                buf = BUF[(tgt, b, rule)]
                if buf['Label'].nunique() < 2:
                    mark(tgt, mname, b, rule, n_labels=len(buf)); continue
                fitb, calb = split_buffer(buf, CFG['thr_frac'], seed)
                nb_cal, _ = label_counts(calb)
                if fitb['Label'].nunique() < 2:
                    mark(tgt, mname, b, rule, n_labels=len(buf), n_fit=len(fitb),
                         n_cal=len(calb), n_benign_cal=nb_cal); continue
                Xf, med = clean_X(fitb, FEATURES)
                Xc, _ = clean_X(calb, FEATURES, medians=med)
                Xe, _ = clean_X(ev, FEATURES, medians=med)
                mdl = make_model(mname, seed, len(Xf))
                t0 = time.time(); mdl.fit(Xf, fitb['Label'].values); fs = round(time.time()-t0, 1)
                p_ev = mdl.predict_proba(Xe)[:, 1]
                p_cal = mdl.predict_proba(Xc)[:, 1]
                ycal = calb['Label'].values

                thr_c, fp_cert, nb_cert, nb_sel = split_certify(ycal, p_cal, CFG['alpha'], seed)
                if np.isnan(thr_c) or nb_cert == 0:
                    mark(tgt, mname, b, rule, n_labels=len(buf), n_fit=len(fitb),
                         n_cal=len(calb), n_benign_cal=nb_cal, n_benign_select=nb_sel,
                         n_benign_certify=nb_cert, fit_s=fs); continue
                bound_c = clopper_pearson_upper(fp_cert, nb_cert, CFG['conf'])
                tpr_c, fpr_c = realised_at_threshold(yev, p_ev, thr_c)
                mcc_c = metrics_at_threshold(yev, p_ev, thr_c)['mcc']

                thr_e = threshold_for_fpr_on_buffer(ycal, p_cal, CFG['alpha'])
                if np.isnan(thr_e):
                    fp_e, bound_b, tpr_e, fpr_e = np.nan, np.nan, np.nan, np.nan
                else:
                    fp_e, n_e = observed_fp(ycal, p_cal, thr_e)
                    bound_b = clopper_pearson_upper(fp_e, n_e, CFG['conf'])
                    tpr_e, fpr_e = realised_at_threshold(yev, p_ev, thr_e)

                mark(tgt, mname, b, rule, n_labels=len(buf), n_fit=len(fitb), n_cal=len(calb),
                     n_benign_cal=nb_cal, n_benign_select=nb_sel, n_benign_certify=nb_cert,
                     fp_certify=fp_cert, thr_split=thr_c, bound_split=bound_c,
                     fpr_split=fpr_c, tpr_split=tpr_c, mcc_split=mcc_c,
                     thr_empirical=thr_e, fp_obs_empirical=fp_e,
                     bound_naive=bound_b, fpr_empirical=fpr_e, tpr_empirical=tpr_e, fit_s=fs)
                del mdl, Xf, Xc, Xe; gc.collect()

print('rows recorded:', len(done))

In [ ]:
V = pd.read_csv(CSV).drop_duplicates(['target','model','budget','rule'])
ok = V[V['bound_split'].notna() & V['fpr_split'].notna()]

print('=== 1. IS THE GUARANTEE HONOURED? ===')
viol = ok[ok['fpr_split'] > ok['bound_split']]
print(f"  cells with a conformal threshold: {len(ok)}")
print(f"  realised FPR above its own bound: {len(viol)} ({len(viol)/max(len(ok),1):.1%}, target <= 5%)")
if len(viol):
    print(viol[['target','model','budget','rule','n_benign_certify','fp_certify','bound_split','fpr_split']].round(4).to_string(index=False))

print('\n=== 2. THE BOUND, PER CORPUS AND BUDGET ===')
t = ok.groupby(['target','budget']).agg(
        n_benign=('n_benign_certify','mean'), fp=('fp_certify','mean'),
        bound=('bound_split','mean'), realised=('fpr_split','mean'),
        tpr=('tpr_split','mean'), mcc=('mcc_split','mean')).round(4)
print(t.to_string())
print('\n  the bound is a statement about the population rate at a threshold chosen by rank,')
print('  and is wide exactly where the calibration sample holds few benign flows.')

print('\n=== 3. CONFORMAL AGAINST THE EMPIRICAL-TARGET CONSTRUCTION ===')
both = ok[ok['fpr_empirical'].notna()]
c = both.groupby('budget').agg(
        fpr_conf=('fpr_split','mean'), fpr_emp=('fpr_empirical','mean'),
        tpr_conf=('tpr_split','mean'), tpr_emp=('tpr_empirical','mean'),
        bound_conf=('bound_split','mean'), bound_binom=('bound_naive','mean'),
        n=('fpr_split','size')).round(4)
print(c.to_string())
ve = both[both['fpr_empirical'] > both['bound_naive']]
print(f"\n  empirical-target thresholds exceeding their Clopper-Pearson bound: {len(ve)} of {len(both)}")
print("  that bound is not valid here, since the same labels chose the threshold; it is shown for comparison only.")

print('\n=== 4. WHAT AN OPERATOR CAN PROMISE AT EACH BUDGET ===')
for b, g in ok.groupby('budget'):
    usable = g[g['bound_split'] <= 0.05]
    print(f"  budget {b}: {len(usable)} of {len(g)} cells can certify a false-positive rate below 5%")
V.round(6).to_csv(f'{RESULT}/fc_conformal_threshold.csv', index=False)
print('\nsaved fc_conformal_threshold.csv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "20: split-certify threshold with a valid false-positive bound"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)